<a href="https://colab.research.google.com/github/LukeMallory/Intro_Predictive_Modeling/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
url = ("https://raw.githubusercontent.com/"
"mwaskom/seaborn-data/master/mpg.csv")
cars = pd.read_csv(url)
cars = cars[
["mpg", "horsepower", "origin"]
].dropna().copy()
cars = cars[
(cars["mpg"] > 0) & (cars["horsepower"] > 0)
]
cars["US"] = (cars["origin"] == "usa").astype(int)
cars["group"] = cars["US"].map({
0: "Non-U.S.",
1: "U.S."
})
colors = {
"Non-U.S.": "navy",
"U.S.": "darkorange"
}
print("Number of cars:", len(cars))
display(cars.head())
display(
cars.groupby("group")["horsepower"].agg(
["min", "max", "count"]
)
)
px.scatter(
cars,
x="horsepower",
y="mpg",
color="group",
color_discrete_map=colors,
title="Horsepower and Fuel Efficiency"
).show()

Number of cars: 392


,mpg,horsepower,origin,US,group
0,18.0,130.0,usa,1,U.S.
1,15.0,165.0,usa,1,U.S.
2,18.0,150.0,usa,1,U.S.
3,16.0,150.0,usa,1,U.S.
4,17.0,140.0,usa,1,U.S.


,min,max,count
group,,,
Non-U.S.,46.0,133.0,147
U.S.,52.0,230.0,245


In [3]:
y = cars["mpg"]
X_A = cars[["horsepower", "US"]]
model_A = LinearRegression()
model_A.fit(X_A, y)
print("Intercept:", model_A.intercept_)
print("Horsepower coefficient:", model_A.coef_[0])
print("US coefficient:", model_A.coef_[1])
print("R-squared:", model_A.score(X_A, y))
cars["predicted_A"] = model_A.predict(X_A)
plot_data = cars.sort_values("horsepower")
fig = px.scatter(
cars,
x="horsepower",
y="mpg",
color="group",
color_discrete_map=colors,
title="Model A: Additive Model"
)
# Draw each group’s fitted line separately.
for group in ["Non-U.S.", "U.S."]:
  group_data = plot_data[
  plot_data["group"] == group
  ]
  fig.add_trace(
    go.Scatter(
    x=group_data["horsepower"],
    y=group_data["predicted_A"],
    mode="lines",
    name=group + " fit",
    line=dict(color=colors[group])
    )
    )
fig.show()

Intercept: 39.8611415042827
Horsepower coefficient: -0.1338131354373828
US coefficient: -3.8973548867229253
R-squared: 0.650491345507443


Part 1:

1a. The relationship between horsepower and mpg appears to be a strong negative relaitonship. The shape of the scatterplot is curved, which is indicative of a nonlinear relationship. A straight line relationship may miss the curvature.
.

1b. The beta 1 coefficient represents horsepower. According to the model, for every 1-horsepower increase, the city-cycle miles per gallon is predicted to increase by beta 1 mpg on average.

The beta 2 coefficient represents the difference between cars from the United States versus cars from Japan and Europe. The indicator variable of U will be 1 for U.S. cars and 0 for the reference group of Japanese and European cars.

1c. Because the fitted group lines are paralell, changes in horsepower are predicted to have the same effect on city-cycle miles per gallon for both groups. Adding the indicator does not allow for different horsepower slopes because this is an additive model.

In [5]:
cars["hp_US"] = cars["horsepower"] * cars["US"]
X_B = cars[["horsepower", "US", "hp_US"]]
model_B = LinearRegression()
model_B.fit(X_B, y)
print("Intercept:", model_B.intercept_)
print("Horsepower coefficient:", model_B.coef_[0])
print("US coefficient:", model_B.coef_[1])
print("Interaction coefficient:", model_B.coef_[2])
print("R-squared:", model_B.score(X_B, y))
cars["predicted_B"] = model_B.predict(X_B)
plot_data = cars.sort_values("horsepower")
fig = px.scatter(
cars,
x="horsepower",
y="mpg",
color="group",
color_discrete_map=colors,
title="Model B: Interaction Model"
)
for group in ["Non-U.S.", "U.S."]:
  group_data = plot_data[
  plot_data["group"] == group
  ]
  fig.add_trace(
    go.Scatter(
    x=group_data["horsepower"],
    y=group_data["predicted_B"],
    mode="lines",
    name=group + " fit",
    line=dict(color=colors[group])
    )
  )
fig.show()

c0 = model_B.intercept_
c1, c2, c3 = model_B.coef_
print("Non-US slope:", c1)
print("US slope:", c1 + c3)
print(
"US minus non-US difference at 100 HP:",
c2 + c3 * 100
)
print(
"US minus non-US difference at 200 HP:",
c2 + c3 * 200
)

Intercept: 47.34071476591622
Horsepower coefficient: -0.22710946716925542
US coefficient: -12.864218766807527
Interaction coefficient: 0.10578943014859377
R-squared: 0.6720427978741312


Non-US slope: -0.22710946716925542
US slope: -0.12132003702066166
US minus non-US difference at 100 HP: -2.2852757519481504
US minus non-US difference at 200 HP: 8.293667262911226


2a.

Model B (U.S.):

c0 + c2 + H(c1 + c3)
Intercept: c0 + c2
Horsepower slope: c1 + c3


Model B (Non-U.S.)

c0 + c1H
Intercept: c0
Horsepower slope: c1



2b.

U.S. Group horsepower slope: -0.12132003702066166

Non-U.S. Horespower slope: Non-US slope: -0.22710946716925542

The b3 coefficient of 0.10578943014859377 represents the predicted difference in city-cycle mpg caused by a 1-horsepower increase between U.S. and non-U.S. cars. The direction of comparison is United States slope minus non-United States slope.

2c.

At 100 horsepower, the predicted difference between U.S. and non-U.S. MPG is 10.58 horsepower, and a 200 the predicted difference is 21.18. The ovservation at 100 horsepower is supported more strongly by the observed horsepower ranges because there are no observed non-U.S. cars that are above 150 horsepower. C2 alone is not either comparison because the interaction term is also dependent on U, and therefore the combination of c2 and c3H represents the difference.

In [7]:
cars["log_hp"] = np.log(cars["horsepower"])
cars["log_mpg"] = np.log(cars["mpg"])
X_C = cars[["log_hp", "US"]]
y_log = cars["log_mpg"]
model_C = LinearRegression()
model_C.fit(X_C, y_log)

print("Intercept:", model_C.intercept_)
print("Log horsepower coefficient:", model_C.coef_[0])
print("US coefficient:", model_C.coef_[1])
print(
"R-squared on log scale:",
model_C.score(X_C, y_log)
)
cars["predicted_log_C"] = model_C.predict(X_C)
plot_data = cars.sort_values("log_hp")
fig = px.scatter(
cars,
x="log_hp",
y="log_mpg",
color="group",
color_discrete_map=colors,
title="Model C: Log-Log Model"
)
for group in ["Non-U.S.", "U.S."]:
  group_data = plot_data[
  plot_data["group"] == group
  ]
  fig.add_trace(
  go.Scatter(
  x=group_data["log_hp"],
  y=group_data["predicted_log_C"],
  mode="lines",
  name=group + " fit",
  line=dict(color=colors[group])
  )
  )
fig.show()
# Convert fitted log MPG back to the original MPG scale.
cars["predicted_C"] = np.exp(cars["predicted_log_C"])
plot_data = cars.sort_values("horsepower")
fig = px.scatter(
cars,
x="horsepower",
y="mpg",
color="group",
color_discrete_map=colors,
title="Model C: Original MPG Scale"
)
for group in ["Non-U.S.", "U.S."]:
  group_data = plot_data[
  plot_data["group"] == group
  ]
  fig.add_trace(
  go.Scatter(
      x=group_data["horsepower"],
  y=group_data["predicted_C"],
  mode="lines",
  name=group + " fit",
  line=dict(color=colors[group])
  )
  )
fig.show()
print(
"Exact percent change for 10% more horsepower:",
100 * (1.10**model_C.coef_[0] - 1)
)

Intercept: 6.631598579391502
Log horsepower coefficient: -0.7538372707536691
US coefficient: -0.11957112714500405
R-squared on log scale: 0.7438527991481687


Exact percent change for 10% more horsepower: -6.932799319991078


3a.

Horsepower and MPG must be positive because you cannot take a natural log of a negative value. You can still fit model c using a linear regression because the relationship between the logs MPG and Horsepower is linear.

3b.

For a 1% increase in horsepower, MPG is predicted to decrease by 0.7538%. This differs from b1 in model A because log coefficients and outcome variables are interpreted in percentages.

3c.

For a 10% increase in horsepower, MPG is preficted to decrease by 6.93%. Model C predicts the same percentage change for 80 to 88 horsepower as for 100 to 110 within a group because the they are both 10% changes. However, this does not predict the same absolute change in MPG because they are different MPG values that receive the same percentage change.

In [8]:
cars["residual_A"] = (
cars["mpg"] - cars["predicted_A"]
)
cars["residual_B"] = (
cars["mpg"] - cars["predicted_B"]
)
cars["residual_log_C"] = (
cars["log_mpg"] - cars["predicted_log_C"]
)
# Model A residual plot.
fig = px.scatter(
cars,
x="predicted_A",
y="residual_A",
color="group",
color_discrete_map=colors,
title="Model A: MPG Residuals"
)
fig.add_hline(y=0, line_dash="dash")
fig.show()
# Model B residual plot.
fig = px.scatter(
cars,
x="predicted_B",
y="residual_B",
color="group",
color_discrete_map=colors,
title="Model B: MPG Residuals"
)
fig.add_hline(y=0, line_dash="dash")
fig.show()
# Model C residual plot on the log scale.
fig = px.scatter(
cars,
x="predicted_log_C",y="residual_log_C",
color="group",
color_discrete_map=colors,
title="Model C: Log-MPG Residuals"
)
fig.add_hline(y=0, line_dash="dash")
fig.show()
# All RMSE values use the original MPG scale.
rmse_A = np.sqrt(
np.mean((y - cars["predicted_A"])**2)
)
rmse_B = np.sqrt(
np.mean((y - cars["predicted_B"])**2)
)
rmse_C = np.sqrt(
np.mean((y - cars["predicted_C"])**2)
)
r2_A = model_A.score(X_A, y)
r2_B = model_B.score(X_B, y)
r2_C = model_C.score(X_C, y_log)
n = len(cars)
fit_table = pd.DataFrame({
"Model": ["A", "B", "C"],
"Response scale": ["MPG", "MPG", "log(MPG)"],
"Predictor columns": [2, 3, 2],
"R2 on fitting scale": [r2_A, r2_B, r2_C],
"Adjusted R2 on fitting scale": [
1 - (1-r2_A)*(n-1)/(n-2-1),
1 - (1-r2_B)*(n-1)/(n-3-1),
1 - (1-r2_C)*(n-1)/(n-2-1)
],
"RMSE in MPG": [rmse_A, rmse_B, rmse_C]
})
display(fit_table.round(4))

,Model,Response scale,Predictor columns,R2 on fitting scale,Adjusted R2 on fitting scale,RMSE in MPG
0,A,MPG,2,0.6505,0.6487,4.6084
1,B,MPG,3,0.6720,0.6695,4.4640
2,C,log(MPG),2,0.7439,0.7425,4.3382


1a.

The model A residual vs fitted plot shows some curvature with the United States residuals. The non-U.S. residuals appear to be relativeky evenly distributed. With lower predicted values, the model tends to under predict MPG, and as the predicted values increase, the spread of over predicting and becomes more even.

Model C does not have any curvature and the vertical spread is more even overall. Because of this, logging does seem to improve the pattern.

1b.

Because model A is predicting MPG and Model C is predicting log(MPG), R^2 is not an applicable mode of comparison.

1c.

Model A RMSE: 4.6084

Model C RMSE: 4.3382

Model C has a lower RMSE value. This alone ,however, can not be used to explain residual behavior because the one model predicts MPG and the other predicts log(MPG). It is also not certain that the models will peform well with new cars because the model was trained on a small sample of cars. The sample may not reflect the larger population of cars.




1d.

As a baseline for further work, I would recommend starting with model C. Although it cannot be compared directly to the other two models with adjusted R^2 and root-mse, the residual vs fitted plot shows a more even distribution of residuals than the other two. Although it cannot be used as a direct comparison, the adjusted R^2 value for model C is very strong. A limitation of this model is the number of variables used. There are almost certainly many other varuables that account for variation in city-cyle MPG besides horsepower and the country that the car was made in.

While all 3 models suggest that changes in horsepower causes changes in city-cycle MPG, it cannot be established that changing horsepower is responsible for the varition in MPG. The model reflects a sample of just over 300 cars, which is small when compared to the population of cars in the three countries. It is also possible that the model suffers from underfitting because there are only two explanitory variables used, so it cannot be said that changes in horsepower cause the predicted change in MPG.